Добейтесь на автокодировщике с 2-мерным скрытым пространством на 3-х цифрах: 0, 1 и 3 – ошибки MSE **< 0.034** на скорости обучения **0.001** на **10-й эпохе**.


In [ ]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

%matplotlib inline


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f'Устройство: {device}')


In [ ]:
def clear_old_images(mask='*.jpg'):
    for item in glob.glob(mask):
        os.remove(item)


def get_selected_mnist(digits):
    data = MNIST(root='mnist_data', train=True, download=True)
    images = data.data.float().div(255).unsqueeze(1)
    labels = data.targets

    selected = torch.zeros_like(labels, dtype=torch.bool)
    for digit in digits:
        selected |= labels == digit

    return images[selected], labels[selected]


def encode_all(net, data, step=512):
    points = []
    net.eval()
    with torch.no_grad():
        for start in range(0, len(data), step):
            batch = data[start:start + step].to(device)
            points.append(net(batch).cpu())
    return torch.cat(points, dim=0).numpy()


def draw_latent(epoch_number, loss_value):
    print('________________________')
    print(f'ЭПОХА: {epoch_number}, loss: {loss_value}')
    print('________________________')

    coords = encode_all(encoder, X_train)
    labels = y_train.numpy()

    plt.figure(dpi=100)
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels, alpha=0.6, s=5)
    plt.legend(*scatter.legend_elements(), loc='upper right', title='Классы')
    plt.title(f'Скрытое пространство после эпохи {epoch_number}')
    plt.xlabel('Признак 1')
    plt.ylabel('Признак 2')
    plt.savefig(f'image_{len(glob.glob("*.jpg"))}.jpg')
    plt.show()


def train_autoencoder(net, loader, epochs=10, lr=0.001):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {'loss': []}

    for epoch in range(1, epochs + 1):
        net.train()
        total_loss = 0.0
        total_items = 0

        for batch_x, in loader:
            batch_x = batch_x.to(device)

            optimizer.zero_grad()
            restored = net(batch_x)
            loss = criterion(restored, batch_x)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_x.size(0)
            total_items += batch_x.size(0)

        epoch_loss = total_loss / total_items
        history['loss'].append(epoch_loss)
        draw_latent(epoch, epoch_loss)

    return history


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(2, 7 * 7 * 64),
            nn.ReLU()
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, z):
        z = self.fc(z)
        z = z.view(-1, 64, 7, 7)
        return self.deconv(z)


class AutoEncoder(nn.Module):
    def __init__(self, encoder_part, decoder_part):
        super().__init__()
        self.encoder_part = encoder_part
        self.decoder_part = decoder_part

    def forward(self, x):
        z = self.encoder_part(x)
        return self.decoder_part(z)


In [ ]:
clear_old_images()

numbers = [0, 1, 3]
X_train, y_train = get_selected_mnist(numbers)

train_data = TensorDataset(X_train)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

encoder = Encoder().to(device)
decoder = Decoder().to(device)
autoencoder = AutoEncoder(encoder, decoder).to(device)

print(f'Размер отфильтрованной обучающей выборки: {tuple(X_train.shape)}')


In [ ]:
print('Старт обучения свёрточного автокодировщика...')

history = train_autoencoder(
    autoencoder,
    train_loader,
    epochs=10,
    lr=0.001
)

final_mse = history['loss'][-1]

print(f'Итоговая ошибка MSE на 10-й эпохе: {final_mse:.5f}')
if final_mse < 0.034:
    print('Условие выполнено: MSE < 0.034.')
else:
    print('Условие не выполнено: MSE не ниже 0.034. Нужно перезапустить обучение или скорректировать модель.')
